### Load data from Google Drive

First, you need to mount your Google Drive to this Colab environment. This will allow the notebook to access files stored in your Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Once your Drive is mounted, you can find the path to your file. A common way is to navigate to the file in your Google Drive, right-click on it, select "Get link", and copy the path. Alternatively, you can use the file browser in the left sidebar of Colab (the folder icon) to navigate your mounted Drive (`/content/drive/My Drive/`) and copy the path to your file.

Here's an example of how to load a CSV file named `my_data.csv` that you might have in your Google Drive's root folder using `pandas`:

In [2]:
import pandas as pd

# Replace 'your_file_path.csv' with the actual path to your CSV file in Google Drive
# For example: '/content/drive/My Drive/my_data.csv'
try:
    file_path = '/content/drive/My Drive/RNN/02) IMDB Dataset.csv' # CHANGE THIS TO YOUR FILE PATH
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Please ensure the file exists and the path is correct.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded data from /content/drive/My Drive/RNN/02) IMDB Dataset.csv


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

,0
review,0
sentiment,0


In [5]:
df.duplicated().sum()

np.int64(418)

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

In [8]:
import re

### Preprocessing

#### COnverting to lowercase

In [9]:
df["review"] = df["review"].str.lower()

#### Removing URLs

In [10]:
def remove_urls(text):
  text = re.sub(r"http\S+","",text)
  return text;

df["review"] = df["review"].apply(remove_urls)


In [11]:
def remove_html(text):
  text = re.sub(r"<.*?>","",text)
  return text
df["review"] = df["review"].apply(remove_html)

#### Removing Punctuation

In [12]:
def remove_punct(text):
  text = re.sub(r"[^\w\s]","",text)
  return text;

df["review"] = df["review"].apply(remove_punct)

#### Removing HTML Tags

In [13]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [14]:
import nltk

In [15]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [17]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [18]:
from nltk.tokenize import sent_tokenize, word_tokenize

data = "All work and no play makes jack dull boy. All work and no play makes jack a dull boy."
words = word_tokenize(data)
print(words)

['All', 'work', 'and', 'no', 'play', 'makes', 'jack', 'dull', 'boy', '.', 'All', 'work', 'and', 'no', 'play', 'makes', 'jack', 'a', 'dull', 'boy', '.']


In [19]:
stop = set(stopwords.words('english'))
stop

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [20]:
words = [ word for word in words if word not in stop]

In [21]:
words

['All',
 'work',
 'play',
 'makes',
 'jack',
 'dull',
 'boy',
 '.',
 'All',
 'work',
 'play',
 'makes',
 'jack',
 'dull',
 'boy',
 '.']

In [22]:
" ".join(words)

'All work play makes jack dull boy . All work play makes jack dull boy .'

#### Remove StopWords

In [23]:
def remove_stopwords(text):
  words = word_tokenize(text)
  words = [ word for word in words if word not in stop]
  return " ".join(words)


df["review"] = df["review"].apply(remove_stopwords)


In [24]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


#### Lemmatization

In [25]:
from nltk.stem import WordNetLemmatizer

In [26]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [27]:
lemma = WordNetLemmatizer()
text = "cats"
lemma.lemmatize(text)

'cat'

In [28]:
def remove_lemma(text):
  tokens = word_tokenize(text)
  words = [lemma.lemmatize(word) for word in tokens]
  return " ".join(words)
df["review"] = df["review"].apply(remove_lemma)

In [29]:
df.head()

,review,sentiment
0,one reviewer mentioned watching 1 oz episode y...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically there family little boy jake think t...,negative
4,petter matteis love time money visually stunni...,positive


#### Sentiment Encoding

In [30]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [31]:
df["sentiment"] = le.fit_transform(df["sentiment"])

In [32]:
df.head()

,review,sentiment
0,one reviewer mentioned watching 1 oz episode y...,1
1,wonderful little production filming technique ...,1
2,thought wonderful way spend time hot summer we...,1
3,basically there family little boy jake think t...,0
4,petter matteis love time money visually stunni...,1


In [33]:
df.dtypes


,0
review,object
sentiment,int64


In [34]:
y = df["sentiment"]

#### Vectorization

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000)

In [36]:
X = vectorizer.fit_transform(df["review"])

In [37]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3877928 stored elements and shape (49582, 5000)>

In [38]:
vectorizer.get_feature_names_out()

array(['10', '100', '1000', ..., 'zero', 'zombie', 'zone'], dtype=object)

### Converting into tensors

In [39]:
from sklearn.model_selection import train_test_split


X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [40]:
import torch
import torch.nn as nn

In [41]:
X_train_tensor = torch.tensor(X_train.toarray(),dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.toarray(),dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [42]:
y_train_tensor

tensor([[0.],
        [0.],
        [1.],
        ...,
        [0.],
        [1.],
        [1.]])

In [43]:
from torch.utils.data import TensorDataset,DataLoader

In [44]:
train_tensor_Dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_tensor_Dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [45]:
train_dataloader = DataLoader(train_tensor_Dataset,batch_size=64,shuffle=True)
test_dataloader = DataLoader(test_tensor_Dataset,batch_size=64)

In [46]:
class RNN(nn.Module):
  def __init__(self,input_size,hidden_size=128,num_layers=1):  # input size ,hidden_size=hidden_layers number,num_layers=number of rnn layer
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)  # batch_First = True means here (seq_len,batch_size,input_size) = (batch_size,seq_len,input_size) for simpler calculations
    self.fc = nn.Linear(hidden_size,1)
  def forward(self,x):
    h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size) #number of layers,batch_size,hidden_size
    out,_ = self.rnn(x,h0)      #output = hidden state of all timestep,final hidden state of all timestep
    out = self.fc(out[:,-1,:])
    out = torch.sigmoid(out)
    return out

In [47]:
import torch.optim as optim
model = RNN(input_size=X_train.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [48]:
epochs = 10

for epoch in range(epochs):
  model.train()

  for xb,yb in train_dataloader:
    optimizer.zero_grad()
    xb = xb.unsqueeze(1)
    outputs = model(xb)
    loss = criterion(outputs,yb)

    loss.backward()
    optimizer.step()
  print(f"Epoch : {epoch}/{epochs} and Loss : {loss.item()}")

Epoch : 0/10 and Loss : 0.26111236214637756
Epoch : 1/10 and Loss : 0.4140314757823944
Epoch : 2/10 and Loss : 0.30643725395202637
Epoch : 3/10 and Loss : 0.28341537714004517
Epoch : 4/10 and Loss : 0.34981128573417664
Epoch : 5/10 and Loss : 0.16334298253059387
Epoch : 6/10 and Loss : 0.13984784483909607
Epoch : 7/10 and Loss : 0.1466866284608841
Epoch : 8/10 and Loss : 0.3187163472175598
Epoch : 9/10 and Loss : 0.1390516608953476


In [51]:
model.eval()
total = 0
correct = 0
with torch.no_grad():
  for xb,yb in test_dataloader:
    xb = xb.unsqueeze(1)
    outputs = model(xb)
    total += yb.size(0)
    correct += (outputs.round()==yb).sum().item()
  print(f"Accuracy : {correct/total}")



Accuracy : 0.8731471211051729
